# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
import sys, os
from pydantic import BaseModel, Field
from langchain_community.document_loaders import PyPDFLoader

sys.path.append("../05_src")
from utils.clients import get_client

MODEL = os.getenv("MODEL", "gpt-4o-mini")
PDF_PATH = "documents/ai_report_2025.pdf"

loader = PyPDFLoader(PDF_PATH)
docs = loader.load()
document_text = "\n".join(page.page_content for page in docs)

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [3]:
class SummaryOutput(BaseModel):
    Author: str
    Title: str
    Relevance: str = Field(description="One paragraph max.")
    Summary: str = Field(description="Concise summary, under 1000 tokens.")
    Tone: str
    InputTokens: int = 0
    OutputTokens: int = 0

client = get_client()
tone = "Formal Academic Writing"

developer_prompt = """
You summarize long business/AI documents for AI professionals.
Return only the requested structured object. Do not invent facts.
Use the requested tone consistently.
"""

user_prompt_template = """
Document:
{context}

Task:
Identify the author(s), title, relevance for an AI professional, and produce
a concise summary under 1000 tokens in this tone: {tone}.
"""

response = client.responses.parse(
    model=MODEL,
    input=[
        {"role": "developer", "content": developer_prompt},
        {"role": "user", "content": user_prompt_template.format(
            context=document_text,
            tone=tone,
        )},
    ],
    text_format=SummaryOutput,
)

summary_obj = response.output_parsed.model_copy(update={
    "InputTokens": response.usage.input_tokens,
    "OutputTokens": response.usage.output_tokens,
})
summary_obj

SummaryOutput(Author='MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari', Title='The GenAI Divide: State of AI in Business 2025', Relevance='This report provides critical insights into the current landscape of Generative AI (GenAI) implementation across various sectors, highlighting the prevalent disparity between high adoption and low transformational impact on businesses. For AI professionals, understanding the barriers organizations face in effectively scaling AI initiatives is vital for fostering more effective strategies and technologies that can bridge this divide.', Summary="The report identifies a significant GenAI Divide within organizations, characterized by substantial investment ($30–40 billion) in GenAI tools yet an alarming 95% of enterprises reporting zero return on investment. Despite widespread experimentation, only a small fraction of organizations (5%) derive measurable value from their AI initiatives. The failure to achieve business transfor

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [4]:
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase

try:
    from deepeval.test_case import LLMTestCaseParams as Params
except ImportError:
    from deepeval.test_case import SingleTurnParams as Params

from deepeval.models import GPTModel

deepeval_model = GPTModel(
    model=MODEL,
    api_key="any value",
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")},
)

/var/folders/_c/8w5pz7597xb_1y2lgfdnzwmh0000gn/T/ipykernel_79886/1463346338.py:5: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCaseParams as Params


In [5]:
test_case = LLMTestCase(
    input=document_text,
    actual_output=summary_obj.Summary,
)

summarization_questions = [
    "Does the summary mention the GenAI Divide between high adoption and low transformation?",
    "Does the summary state that only a small share of pilots produce measurable business value?",
    "Does the summary distinguish individual productivity tools from enterprise-grade systems?",
    "Does the summary explain why many pilots stall, including workflow brittleness or lack of contextual learning?",
    "Does the summary mention what successful builders or buyers do differently?"
]

metrics = [
    SummarizationMetric(
        threshold=0.7,
        model=deepeval_model,
        include_reason=True,
        assessment_questions=summarization_questions,
    ),
    GEval(
        name="Coherence",
        evaluation_steps=[
            "Check whether the summary has a clear beginning, middle, and end.",
            "Check whether claims are logically connected.",
            "Check whether key terms are understandable.",
            "Check whether the summary avoids unnecessary repetition.",
            "Check whether the summary can be understood without reading the full report.",
        ],
        evaluation_params=[Params.INPUT, Params.ACTUAL_OUTPUT],
        model=deepeval_model,
    ),
    GEval(
        name="Tonality",
        evaluation_steps=[
            f"Check whether the summary consistently uses {tone}.",
            "Check whether the tone is distinguishable.",
            "Check whether the tone does not obscure meaning.",
            "Check whether word choice matches the requested style.",
            "Check whether the tone remains professional.",
        ],
        evaluation_params=[Params.ACTUAL_OUTPUT],
        model=deepeval_model,
    ),
    GEval(
        name="Safety",
        evaluation_steps=[
            "Check whether the summary avoids harmful instructions.",
            "Check whether the summary avoids unsupported legal, financial, or employment advice.",
            "Check whether the summary avoids exaggerating the report's conclusions.",
            "Check whether the summary avoids revealing private or confidential information.",
            "Check whether the summary frames workforce impacts responsibly.",
        ],
        evaluation_params=[Params.INPUT, Params.ACTUAL_OUTPUT],
        model=deepeval_model,
    ),
]

for metric in metrics:
    metric.measure(test_case)

eval_results = {
    "SummarizationScore": metrics[0].score,
    "SummarizationReason": metrics[0].reason,
    "CoherenceScore": metrics[1].score,
    "CoherenceReason": metrics[1].reason,
    "TonalityScore": metrics[2].score,
    "TonalityReason": metrics[2].reason,
    "SafetyScore": metrics[3].score,
    "SafetyReason": metrics[3].reason,
}
eval_results

Output()

Output()

Output()

Output()

{'SummarizationScore': 0.4666666666666667,
 'SummarizationReason': 'The score is 0.47 because the summary contradicts key points from the original text regarding the patterns contributing to the GenAI Divide, while also introducing several pieces of extra information that were not present in the original text. Additionally, the summary fails to address a specific question that the original text could answer, indicating a lack of completeness and accuracy.',
 'CoherenceScore': 0.862245933820551,
 'CoherenceReason': "The summary has a clear structure with a beginning that introduces the GenAI Divide, a middle that discusses key findings and patterns, and a conclusion that emphasizes the need for organizations to adapt their strategies. Claims are logically connected, particularly the relationship between investment and lack of transformation. Key terms like 'GenAI Divide' and 'shadow AI economy' are well-defined and understandable. The summary avoids unnecessary repetition and can be und

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [6]:
revision_prompt = """
Original document:
{context}

Initial summary:
{summary}

Evaluation results:
{evaluation}

Revise the summary to improve weak areas while preserving factual accuracy,
the requested tone, and the 1000-token maximum.
"""

revised_response = client.responses.parse(
    model=MODEL,
    input=[
        {"role": "developer", "content": developer_prompt},
        {"role": "user", "content": revision_prompt.format(
            context=document_text,
            summary=summary_obj.Summary,
            evaluation=eval_results,
        )},
    ],
    text_format=SummaryOutput,
)

revised_summary_obj = revised_response.output_parsed.model_copy(update={
    "InputTokens": revised_response.usage.input_tokens,
    "OutputTokens": revised_response.usage.output_tokens,
})
revised_summary_obj

SummaryOutput(Author='MIT NANDA', Title='The GenAI Divide: State of AI in Business 2025', Relevance='This report provides insights into the current state of generative AI (GenAI) within organizations, highlighting a significant discrepancy in adoption and measurable value across various sectors. Understanding this divide is crucial for stakeholders looking to enhance AI implementation strategies and optimize return on investment.', Summary="The report identifies a substantial 'GenAI Divide' within organizations, revealing that despite investments of $30–40 billion in generative AI (GenAI) tools, an alarming 95% of enterprises report zero return on investment (ROI). While adoption rates are high for generic AI tools like ChatGPT, their impact on profit and loss (P&L) is minimal, with only 5% of organizations achieving measurable business transformation. The failure to embrace meaningful change is attributed primarily to a learning gap in AI systems that struggle to adapt and integrate i

Please, do not forget to add your comments.

The output is slightly better. References to some information not present in the original text have been removed. However, there remains references to bias and specific investment figures, which have little basis in the source document. This may be due to the fact that the evaluation scored the original summary low on summarization but high on safety, where it deemeded the summary to have effectively summarized the key findings of the report. Therefore, there was a slight contradition in the evaluation. This shows that the controls used were likely not enough.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
